# Estudio de mercado CAM – Versión **gratuita** (DENUE + OpenStreetMap + OSRM + Folium)

**Objetivo:** evaluar el potencial de mercado para instalar un **CAM (Centro de Atención Médica)** en dos ubicaciones
(San Lucas y San José), considerando como zona de influencia los establecimientos a **≤ 30 minutos de traslado**
(se mide en **tiempo**, no en kilómetros).

| # | Categoría | Subtemas (color en el mapa) |
|---|---|---|
| 1 | CAM de Consulta Externa 🏥 | Clínica / centro médico · Consultorios agrupados · Consultorio adyacente a farmacia · Consultorio · Hospital (referencia) |
| 2 | Clínicas de Especialidad 🩺 | Ginecología · Pediatría · Medicina Interna · Traumatología · Cardiología · Dermatología · Oftalmología |
| 3 | Laboratorios Clínicos 🧪 | **Cadena** – Chopo · **Cadena** – Salud Digna · **Cadena** – Otra nacional · **Local** / independiente |
| 4 | Gabinetes de Imagenología 🩻 | Rayos X · Ultrasonido · Mastografía · Tomografía · Resonancia |

**Salidas por ubicación:** 5 mapas (1 general + 1 por categoría), con icono por categoría, color por subtema y
capas que se pueden encender/apagar; y un Excel con una hoja por categoría + resumen por bandas de tiempo.
Además, un Excel comparativo San Lucas vs San José.

> Datos que ninguna fuente publica de forma sistemática (número de consultorios, número de médicos) quedan
> marcados como *"validar en campo"*.

**Fuentes gratuitas:**
- **DENUE (INEGI)**: directorio oficial de establecimientos de México (nombre, clase SCIAN, dirección, teléfono y
  *personal ocupado*). Requiere un token gratuito: <https://www.inegi.org.mx/servicios/api_denue.html>
  (variable de entorno `DENUE_TOKEN`). Sin token se usa solo OpenStreetMap.
- **OpenStreetMap / Overpass**: complemento (a veces incluye horarios y especialidad).
- **OSRM**: tiempo de traslado en auto (sin tráfico; ajustable con `FACTOR_TRAFICO`) y zona isócrona ≤ 30 min.
- **Folium / Leaflet**: mapas interactivos sin costo.

In [ ]:
%pip install -q requests pandas openpyxl folium

## 1. Configuración

In [ ]:
import os
import re, math, json, html, time, unicodedata
from pathlib import Path
from urllib.parse import unquote
import requests
import pandas as pd
DENUE_TOKEN = os.getenv("DENUE_TOKEN", "")   # token gratuito INEGI: https://www.inegi.org.mx/servicios/api_denue.html
FACTOR_TRAFICO = 1.0     # OSRM no considera tráfico; p. ej. 1.25 = +25 % en hora pico
ISOCRONA = True          # dibuja la zona ≤ 30 min (malla de puntos evaluada con OSRM)
PASO_MALLA_M = 1_500
OSRM = "https://router.project-osrm.org"      # servidor público de demostración (uso moderado)
OVERPASS = "https://overpass-api.de/api/interpreter"
SALIDA = Path("salidas_gratuito"); SALIDA.mkdir(exist_ok=True)

# Ubicaciones de estudio: el enlace corto se resuelve automáticamente a coordenadas.
# Si tu red no permite resolverlo, escribe lat/lng manualmente (clic derecho en Google Maps → copiar coordenadas).
UBICACIONES = {
    "San Lucas": {"url": "https://maps.app.goo.gl/K6HAcsMj4TUccqUU7", "lat": None, "lng": None},
    "San José":  {"url": "https://maps.app.goo.gl/mVBjCMaCyxbyPQDu7", "lat": None, "lng": None},
}

TIEMPO_MAX_MIN = 30          # zona de influencia: tiempo de traslado máximo al sitio (minutos)
BANDAS_MIN = [10, 20, 30]    # bandas de tiempo para colorear / resumir
RADIO_BUSQUEDA_M = 25_000    # radio de búsqueda previo (se filtra después por TIEMPO, no por km)

## 2. Catálogo de categorías, subtemas, colores y resolución de ubicaciones

In [ ]:
def norm(txt):
    """minúsculas y sin acentos, para comparar palabras clave."""
    txt = unicodedata.normalize("NFKD", str(txt or "")).encode("ascii", "ignore").decode()
    return " " + re.sub(r"\s+", " ", txt.lower()) + " "

def contiene(texto, claves):
    """True si alguna clave aparece en el texto (los espacios en la clave marcan límite de palabra)."""
    t = norm(texto)
    return any(norm(c)[1:-1] in t for c in claves)

CAT_CAM, CAT_ESP, CAT_LAB, CAT_IMG = ("CAM Consulta Externa", "Clínica de Especialidad",
                                      "Laboratorio Clínico", "Gabinete de Imagenología")
CATEGORIAS = [CAT_CAM, CAT_ESP, CAT_LAB, CAT_IMG]

# Especialidades solicitadas (claves en español e inglés/OSM)
ESPECIALIDADES = {
    "Ginecología":       ["ginecolog", "gineco", "obstetr", "gynaecolog", "gynecolog", "maternidad"],
    "Pediatría":         ["pediatr", "paediatr", "pediatric", "ninos", "infantil"],
    "Medicina Interna":  ["medicina interna", "internista", "internal"],
    "Traumatología":     ["traumatolog", "ortoped", "orthopaed", "orthoped", "trauma"],
    "Cardiología":       ["cardiolog", "cardio", "corazon"],
    "Dermatología":      ["dermatolog", "dermato", "piel"],
    "Oftalmología":      ["oftalmolog", "ophthalmolog", "retina", "ojos"],
}

# Cadenas de laboratorio (subtema = cadena vs local)
CADENAS_LAB = {
    "Chopo":        ["chopo"],
    "Salud Digna":  ["salud digna"],
    "Otra nacional": ["olab", "laboratorio medico polanco", "lab polanco", "jenner", "laboratorios azteca",
                      "biomedica de referencia", "carpermor", " lapi ", "clinicos de puebla", "laboratorios ruiz",
                      "similares", "moreira", "diagnostico clinico hispano", "unilabs", "biolab", "labclinic",
                      "laboratorios del chopo", "orthin", "medica sur"],
}

# Estudios de imagen, del más complejo al más básico (el más complejo define el subtema)
ESTUDIOS_IMAGEN = {
    "Resonancia":   ["resonancia", " mri ", " rm "],
    "Tomografía":   ["tomograf", " tac ", "ct scan", "tomography"],
    "Mastografía":  ["mastograf", "mamograf", "mammogra"],
    "Ultrasonido":  ["ultrasonido", "ultrasound", "ecograf", "sonograf", " eco "],
    "Rayos X":      ["rayos x", "rayos-x", "x-ray", "xray", "radiograf", "radiolog"],
}
CLAVES_IMAGEN = sum(ESTUDIOS_IMAGEN.values(), []) + ["imagen", "imagenolog", "diagnostico por imagen", "imaging", "radiology"]
CLAVES_LAB = ["laborator", "analisis clinic", "lab ", "patolog", "medical_lab", "laboratory"]

SERVICIOS = {
    "Urgencias": ["urgencia", "emergenc"], "24 horas": ["24 h", "24 horas", "24/7", "open 24"],
    "Hospitalización": ["hospital"], "Farmacia": ["farmacia", "pharmacy"],
    "Laboratorio": CLAVES_LAB, "Imagen": CLAVES_IMAGEN, "Toma a domicilio": ["domicilio", "home service"],
    "Check-up": ["check up", "checkup", "chequeo"], "Consulta general": ["medicina general", "medico general"],
}

EXCLUIR = ["veterinar", "dental", "dentist", "odontolog", "ortodonc", " spa ", "estetica", "belleza", "psicolog",
           "nutriolog", "optica", "optometr", "quiropract", "fisioterap", "computo", "idiomas", "escuela", "pet "]
FARMACIAS = ["farmacia", "similares", "del ahorro", "benavides", "guadalajara", "san pablo", "farmapronto",
             "yza", "pharmacy"]

SUBTEMAS = {  # color HEX por subtema (mismo color en mapa, leyenda y Excel)
    CAT_CAM: {"Clínica / centro médico": "#2E86AB", "Consultorios agrupados / torre médica": "#1B4F72",
              "Consultorio adyacente a farmacia": "#5DADE2", "Consultorio médico": "#85C1E9",
              "Hospital (referencia, con hospitalización)": "#7F8C8D"},
    CAT_ESP: {"Ginecología": "#C2185B", "Pediatría": "#F06292", "Medicina Interna": "#8E24AA",
              "Traumatología": "#5E35B1", "Cardiología": "#D32F2F", "Dermatología": "#FF8A65",
              "Oftalmología": "#6D4C41", "Otra especialidad": "#BDBDBD"},
    CAT_LAB: {"Cadena – Chopo": "#00897B", "Cadena – Salud Digna": "#43A047",
              "Cadena – Otra nacional": "#9CCC65", "Local / independiente": "#F9A825"},
    CAT_IMG: {"Resonancia": "#212121", "Tomografía": "#546E7A", "Mastografía": "#AD1457",
              "Ultrasonido": "#0277BD", "Rayos X": "#FF6F00", "Imagen (tipo no publicado)": "#A1887F"},
}
ICONOS = {CAT_CAM: "house-medical", CAT_ESP: "user-doctor", CAT_LAB: "flask", CAT_IMG: "x-ray"}   # Font Awesome 6
EMOJIS = {CAT_CAM: "🏥", CAT_ESP: "🩺", CAT_LAB: "🧪", CAT_IMG: "🩻"}                              # Google Maps

def detectar(texto, dic):
    return [k for k, claves in dic.items() if contiene(texto, claves)]

def resolver_link(url):
    """Sigue el enlace corto de Google Maps y extrae (lat, lng)."""
    r = requests.get(url, allow_redirects=True, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
    texto = unquote(r.url + " " + r.text[:300_000])
    for patron in (r"!3d(-?\d+\.\d+)!4d(-?\d+\.\d+)", r"@(-?\d+\.\d+),(-?\d+\.\d+)",
                   r"center=(-?\d+\.\d+),(-?\d+\.\d+)", r"[?&]q=(-?\d+\.\d+),(-?\d+\.\d+)",
                   r"ll=(-?\d+\.\d+),(-?\d+\.\d+)"):
        m = re.search(patron, texto)
        if m:
            return float(m.group(1)), float(m.group(2))
    raise ValueError(f"No se encontraron coordenadas en {r.url}")

def haversine_m(lat1, lng1, lat2, lng2):
    p = math.pi / 180
    a = (math.sin((lat2 - lat1) * p / 2) ** 2
         + math.cos(lat1 * p) * math.cos(lat2 * p) * math.sin((lng2 - lng1) * p / 2) ** 2)
    return 12_742_000 * math.asin(math.sqrt(a))

def banda(minutos):
    ini = 0
    for fin in BANDAS_MIN:
        if minutos <= fin:
            return f"{ini}-{fin} min"
        ini = fin
    return f"> {BANDAS_MIN[-1]} min"

for nombre, u in UBICACIONES.items():
    if u["lat"] is None:
        try:
            u["lat"], u["lng"] = resolver_link(u["url"])
        except Exception as e:
            raise SystemExit(f"⚠️ No pude resolver el enlace de {nombre}: {e}. Escribe lat/lng en UBICACIONES.")
    print(f"{nombre}: {u['lat']:.6f}, {u['lng']:.6f}")

## 3. Reglas de clasificación por subtema

In [ ]:
def subtema_lab(nombre):
    """(subtema, marca): la marca identifica la cadena concreta para contar sus sucursales."""
    for cadena, claves in CADENAS_LAB.items():
        for c in claves:
            if contiene(nombre, [c]):
                return f"Cadena – {cadena}", (cadena if cadena != "Otra nacional" else c.strip().title())
    return "Local / independiente", ""

def subtema_img(texto):
    estudios = detectar(texto, ESTUDIOS_IMAGEN)
    return (estudios[0] if estudios else "Imagen (tipo no publicado)"), estudios

def subtema_cam(nombre, texto):
    if contiene(nombre, ["hospital", "sanatorio"]) and not contiene(nombre, ["hospital de dia"]):
        return "Hospital (referencia, con hospitalización)"
    if contiene(nombre, FARMACIAS):
        return "Consultorio adyacente a farmacia"
    if contiene(texto, ["torre medica", "plaza medica", "consultorios", "edificio medico", "medical tower", "medical plaza"]):
        return "Consultorios agrupados / torre médica"
    if contiene(texto, ["clinica", "clinic", "centro medico", "medical center", "medical centre", "policlinica"]):
        return "Clínica / centro médico"
    return "Consultorio médico"

def clasificar(categoria, nombre, texto_extra="", pista=None):
    """Devuelve (subtema, detalle) o None si el registro no pertenece a la categoría."""
    texto = f"{nombre} {texto_extra}"
    if contiene(texto, EXCLUIR) and not contiene(texto, CLAVES_LAB + CLAVES_IMAGEN + ["clinica", "medic"]):
        return None
    if contiene(nombre, ["dental", "odontolog", "veterinar"]):
        return None
    if categoria == CAT_LAB:
        sub, cadena = subtema_lab(nombre)
        if not cadena and not contiene(texto, CLAVES_LAB):
            return None
        if contiene(nombre, ["laboratorio dental", "protesis"]):
            return None
        return sub, {"Cadena": cadena or "—"}
    if categoria == CAT_IMG:
        sub, estudios = subtema_img(texto)
        if pista and pista not in estudios:
            estudios = estudios + [pista]
            if sub.startswith("Imagen"):
                sub = pista
        if not estudios and not contiene(texto, CLAVES_IMAGEN):
            return None
        orden = list(ESTUDIOS_IMAGEN)
        estudios = sorted(set(estudios), key=orden.index)
        return (estudios[0] if estudios else sub), {"Tipos de estudio": ", ".join(estudios) or "No publicado"}
    if categoria == CAT_ESP:
        esp = detectar(texto, ESPECIALIDADES)
        if pista and pista not in esp:
            esp = [pista] + esp
        if contiene(nombre, ["hospital"]):
            return None
        sub = esp[0] if esp else "Otra especialidad"
        return sub, {"Especialidad": ", ".join(esp) or "No identificada"}
    # CAM consulta externa: descarta lo que es solo laboratorio / imagen
    if (contiene(nombre, ["laborator", "rayos x", "ultrasonido", "radiolog", "imagenolog"])
            and not contiene(nombre, ["clinica", "centro medico", "consultorio", "hospital"])):
        return None
    return subtema_cam(nombre, texto), {}

def enriquecer(df):
    """Columnas específicas que pide el estudio, calculadas por categoría."""
    if df.empty:
        return df
    df = df.copy()
    texto = (df["Nombre"].fillna("") + " " + df["Descripción"].fillna("") + " " + df["Tipos"].fillna(""))
    df["Servicios detectados"] = texto.apply(lambda t: ", ".join(detectar(t, SERVICIOS)) or "No publicado")
    df["Especialidades ofrecidas"] = texto.apply(lambda t: ", ".join(detectar(t, ESPECIALIDADES)) or "No publicado")
    labs = df[df["Categoría"] == CAT_LAB][["Latitud", "Longitud"]].values
    imgs = df[df["Categoría"] == CAT_IMG][["Latitud", "Longitud"]].values

    def en_sitio(row, puntos, claves):
        if contiene(f"{row['Nombre']} {row['Descripción']} {row['Tipos']}", claves):
            return "Sí (publicado)"
        if any(haversine_m(row["Latitud"], row["Longitud"], la, ln) < 40 for la, ln in puntos):
            return "Probable (mismo inmueble)"
        return "No detectado"
    df["Laboratorio en sitio"] = df.apply(lambda r: en_sitio(r, labs, CLAVES_LAB), axis=1)
    df["Imagen en sitio"] = df.apply(lambda r: en_sitio(r, imgs, CLAVES_IMAGEN), axis=1)
    # sucursales de la misma cadena dentro de la zona
    marca = [subtema_lab(n)[1] if c == CAT_LAB else "" for n, c in zip(df["Nombre"], df["Categoría"])]
    conteo = pd.Series(1, index=pd.MultiIndex.from_arrays([df["Ubicación estudio"], marca])).groupby(level=[0, 1]).size()
    df["Sucursales en la zona"] = [(conteo[(u, m)] if m else 1) if c == CAT_LAB else None
                                   for u, c, m in zip(df["Ubicación estudio"], df["Categoría"], marca)]
    df["Banda de tiempo"] = df["Tiempo (min)"].apply(banda)
    df["Color subtema"] = df.apply(lambda r: SUBTEMAS[r["Categoría"]].get(r["Subtema"], "#999999"), axis=1)
    return df.sort_values(["Ubicación estudio", "Categoría", "Tiempo (min)"]).reset_index(drop=True)

## 4. Búsqueda de establecimientos (DENUE + OSM) y tiempo de traslado (OSRM)
Se busca en un radio amplio y **después se filtra por tiempo ≤ 30 min**.

In [ ]:
# ---------- Fuente 1: DENUE (INEGI) – directorio oficial de negocios de México ----------
CLASES_DENUE = {   # texto de "Clase_actividad" (SCIAN) → categoría(s) candidatas
    "laboratorios medicos": [CAT_LAB, CAT_IMG], "consultorios de medicina general": [CAT_CAM],
    "consultorios de medicina especializada": [CAT_ESP], "clinicas de consultorios medicos": [CAT_CAM],
    "no requieren hospitalizacion": [CAT_CAM], "hospitales generales": [CAT_CAM],
    "otros consultorios para el cuidado de la salud": [CAT_CAM],
}

def malla(lat, lng, radio, paso):
    dlat = paso / 111_320
    dlng = paso / (111_320 * math.cos(math.radians(lat)))
    n = int(radio // paso) + 1
    return [(lat + i * dlat, lng + j * dlng) for i in range(-n, n + 1) for j in range(-n, n + 1)
            if haversine_m(lat, lng, lat + i * dlat, lng + j * dlng) <= radio]

def buscar_denue(lat, lng, radio):
    if not DENUE_TOKEN:
        print("⚠️ Sin DENUE_TOKEN: se omite DENUE (se usará solo OpenStreetMap)")
        return []
    vistos, filas = set(), []
    centros = malla(lat, lng, radio, 7_000)                 # DENUE busca en radios de máximo 5 km
    for k, (la, ln) in enumerate(centros, 1):
        url = f"https://www.inegi.org.mx/app/api/denue/v1/consulta/Buscar/todos/{la:.6f},{ln:.6f}/5000/{DENUE_TOKEN}"
        try:
            data = requests.get(url, timeout=90).json()
        except Exception as e:
            print("DENUE error:", e); continue
        for d in data if isinstance(data, list) else []:
            if d.get("Id") in vistos:
                continue
            vistos.add(d.get("Id"))
            clase = norm(d.get("Clase_actividad"))
            cats = next((v for k2, v in CLASES_DENUE.items() if k2 in clase), None)
            if cats:
                filas.append((d, cats))
        print(f"\rDENUE {k}/{len(centros)} – {len(filas)} registros de salud", end="")
    print()
    return filas

def registros_denue(nombre_ubi, u):
    out = []
    for d, cats in buscar_denue(u["lat"], u["lng"], RADIO_BUSQUEDA_M):
        nombre = d.get("Nombre") or d.get("Razon_social") or ""
        extra = f"{d.get('Clase_actividad', '')} {d.get('Razon_social', '')}"
        if CAT_LAB in cats:   # DENUE junta laboratorios e imagen en la misma clase: separa por nombre
            cats = [c for c, cl in ((CAT_LAB, CLAVES_LAB + sum(CADENAS_LAB.values(), [])), (CAT_IMG, CLAVES_IMAGEN))
                    if contiene(nombre, cl)] or [CAT_LAB]
        for cat in cats:
            clas = clasificar(cat, nombre, extra if cat != CAT_CAM else d.get("Razon_social", ""))
            if clas is None:
                continue
            sub, det = clas
            clase = norm(d.get("Clase_actividad"))
            if cat == CAT_CAM and "hospitales" in clase:
                sub = "Hospital (referencia, con hospitalización)"
            elif cat == CAT_CAM and "clinicas de consultorios" in clase and sub in ("Consultorio médico", "Clínica / centro médico"):
                sub = "Consultorios agrupados / torre médica"
            direccion = " ".join(str(d.get(c) or "") for c in ["Tipo_vialidad", "Calle", "Num_Exterior", "Colonia", "CP", "Ubicacion"])
            la, ln = float(d["Latitud"]), float(d["Longitud"])
            out.append({"Ubicación estudio": nombre_ubi, "Categoría": cat, "Subtema": sub, "Nombre": nombre.title(),
                        "Dirección": re.sub(r"\s+", " ", direccion).strip(), "Latitud": la, "Longitud": ln,
                        "Horarios": "No publicado", "Teléfono": d.get("Telefono", ""), "Sitio web": d.get("Sitio_internet", ""),
                        "Calificación": None, "Reseñas": None, "Descripción": d.get("Clase_actividad", ""), "Tipos": "",
                        "Link mapa": f"https://www.openstreetmap.org/?mlat={la}&mlon={ln}#map=18/{la}/{ln}",
                        "Número de consultorios": "No publicado (validar en campo)",
                        "Número aproximado de médicos": f"Personal ocupado DENUE: {d.get('Estrato', 's/d')}",
                        "Fuente": "DENUE INEGI", **det})
    return out

# ---------- Fuente 2: OpenStreetMap (Overpass) ----------
def buscar_osm(lat, lng, radio):
    a = f"(around:{radio},{lat},{lng})"
    q = f"""[out:json][timeout:180];(
      nwr{a}["amenity"~"^(clinic|doctors|hospital)$"];
      nwr{a}["healthcare"];
      nwr{a}["name"~"laborator|rayos|ultrason|radiolog|imagen|tomogra|resonan|mastogra|consultori|cl[ií]nica|m[eé]dic",i];
    );out center tags;"""
    r = requests.post(OVERPASS, data={"data": q}, timeout=240, headers={"User-Agent": "estudio-mercado-cam"})
    r.raise_for_status()
    return r.json().get("elements", [])

def registros_osm(nombre_ubi, u):
    out = []
    for e in buscar_osm(u["lat"], u["lng"], RADIO_BUSQUEDA_M):
        t = e.get("tags", {})
        nombre = t.get("name") or t.get("brand") or ""
        if not nombre:
            continue
        la, ln = (e.get("lat"), e.get("lon")) if "lat" in e else (e["center"]["lat"], e["center"]["lon"])
        hc, am, esp = t.get("healthcare", ""), t.get("amenity", ""), t.get("healthcare:speciality", "")
        texto = f"{nombre} {esp} {hc} {am} {t.get('description', '')}"
        cats = []
        if hc == "laboratory" or contiene(nombre, CLAVES_LAB) or subtema_lab(nombre)[1]:
            cats.append(CAT_LAB)
        if contiene(texto, CLAVES_IMAGEN) or "radiology" in esp:
            cats.append(CAT_IMG)
        if detectar(texto, ESPECIALIDADES):
            cats.append(CAT_ESP)
        if am in ("clinic", "hospital") or hc in ("clinic", "hospital", "centre") or (am == "doctors" and not cats):
            cats.append(CAT_CAM)
        for cat in cats:
            clas = clasificar(cat, nombre, texto)
            if clas is None:
                continue
            sub, det = clas
            dire = " ".join(t.get(k, "") for k in ["addr:street", "addr:housenumber", "addr:suburb", "addr:postcode", "addr:city"])
            out.append({"Ubicación estudio": nombre_ubi, "Categoría": cat, "Subtema": sub, "Nombre": nombre,
                        "Dirección": dire.strip() or "Ver mapa", "Latitud": la, "Longitud": ln,
                        "Horarios": t.get("opening_hours", "No publicado"), "Teléfono": t.get("phone", t.get("contact:phone", "")),
                        "Sitio web": t.get("website", t.get("contact:website", "")), "Calificación": None, "Reseñas": None,
                        "Descripción": t.get("description", ""), "Tipos": f"{am} {hc} {esp}".strip(),
                        "Link mapa": f"https://www.openstreetmap.org/{e['type']}/{e['id']}",
                        "Número de consultorios": "No publicado (validar en campo)",
                        "Número aproximado de médicos": "No publicado (validar en campo)",
                        "Fuente": "OpenStreetMap", **det})
    return out

def deduplicar(df):
    """Une DENUE + OSM: mismo nombre (primeras 3 palabras) a < 100 m dentro de la misma categoría."""
    df = df.assign(_k=df["Nombre"].apply(lambda s: " ".join(norm(s).split()[:3]))).sort_values("Fuente")
    keep = []
    for _, g in df.groupby(["Ubicación estudio", "Categoría", "_k"]):
        elegidos = []
        for i, r in g.iterrows():
            if all(haversine_m(r["Latitud"], r["Longitud"], df.at[j, "Latitud"], df.at[j, "Longitud"]) > 100 for j in elegidos):
                elegidos.append(i)
        keep += elegidos
    return df.loc[keep].drop(columns="_k")

# ---------- Tiempo de traslado: OSRM (gratuito) ----------
def tiempos_osrm(origenes, destino, lote=90):
    """Minutos y km desde cada origen hasta el destino. Descarta orígenes a > 500 m de una calle (p. ej., mar)."""
    res = []
    for ini in range(0, len(origenes), lote):
        pts = [destino] + origenes[ini:ini + lote]
        coords = ";".join(f"{ln:.6f},{la:.6f}" for la, ln in pts)
        url = (f"{OSRM}/table/v1/driving/{coords}?sources={';'.join(map(str, range(1, len(pts))))}"
               f"&destinations=0&annotations=duration,distance")
        data = requests.get(url, timeout=120).json()
        if data.get("code") != "Ok":
            print("OSRM:", data.get("message", data.get("code")))
            res += [(None, None)] * (len(pts) - 1); continue
        for k, fila in enumerate(data.get("durations", [])):
            snap = data["sources"][k].get("distance", 0)
            dur, dist = fila[0], data["distances"][k][0]
            res.append((None, None) if dur is None or snap > 500 else (dur / 60 * FACTOR_TRAFICO, dist / 1000))
        time.sleep(1)
    return res

def recolectar(nombre_ubi, u):
    filas = registros_denue(nombre_ubi, u)
    try:
        filas += registros_osm(nombre_ubi, u)
    except Exception as e:
        print("Overpass error:", e)
    df = pd.DataFrame(filas)
    if df.empty:
        return df
    df = deduplicar(df)
    t = tiempos_osrm(list(zip(df["Latitud"], df["Longitud"])), (u["lat"], u["lng"]))
    df["Tiempo (min)"] = [round(x[0], 1) if x[0] is not None else None for x in t]
    df["Distancia ruta (km)"] = [round(x[1], 2) if x[1] is not None else None for x in t]
    df = df[df["Tiempo (min)"].notna() & (df["Tiempo (min)"] <= TIEMPO_MAX_MIN)]
    print(f"{nombre_ubi}: {len(df)} establecimientos a ≤ {TIEMPO_MAX_MIN} min")
    return df

datos = pd.concat([recolectar(n, u) for n, u in UBICACIONES.items()], ignore_index=True)
assert not datos.empty, "No se encontraron establecimientos: revisa claves de API y coordenadas"
datos = enriquecer(datos)
datos.groupby(["Ubicación estudio", "Categoría", "Subtema"]).size().to_frame("Establecimientos")

## 5. Tablas de Excel

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

COLUMNAS_BASE = ["Ubicación estudio", "Categoría", "Subtema", "Nombre", "Dirección", "Tiempo (min)",
                 "Banda de tiempo", "Distancia ruta (km)", "Horarios", "Teléfono", "Sitio web", "Calificación",
                 "Reseñas", "Latitud", "Longitud", "Link mapa", "Fuente"]
COLUMNAS_CAT = {
    CAT_CAM: ["Número de consultorios", "Especialidades ofrecidas", "Laboratorio en sitio", "Imagen en sitio",
              "Servicios detectados"],
    CAT_ESP: ["Especialidad", "Número aproximado de médicos", "Servicios detectados"],
    CAT_LAB: ["Cadena", "Sucursales en la zona", "Servicios detectados"],
    CAT_IMG: ["Tipos de estudio", "Servicios detectados"],
}
HOJAS = {CAT_CAM: "1_CAM_Consulta_Externa", CAT_ESP: "2_Clinicas_Especialidad",
         CAT_LAB: "3_Laboratorios", CAT_IMG: "4_Imagenologia"}

def _formatear(ws, df, col_color=None):
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    for c in ws[1]:
        c.font = Font(bold=True, color="FFFFFF")
        c.fill = PatternFill("solid", fgColor="1F3A5F")
        c.alignment = Alignment(wrap_text=True, vertical="center")
    for i, col in enumerate(df.columns, 1):
        largo = max([len(str(col))] + [len(str(v)) for v in df[col].head(200)])
        ws.column_dimensions[get_column_letter(i)].width = min(max(10, largo + 2), 60)
    if col_color is not None and "Subtema" in df.columns:
        j = list(df.columns).index("Subtema") + 1
        for fila, color in enumerate(col_color, 2):
            ws.cell(fila, j).fill = PatternFill("solid", fgColor=color.lstrip("#"))
            ws.cell(fila, j).font = Font(color="FFFFFF", bold=True)
    if "Link mapa" in df.columns:
        j = list(df.columns).index("Link mapa") + 1
        for fila in range(2, len(df) + 2):
            celda = ws.cell(fila, j)
            if str(celda.value).startswith("http"):
                celda.hyperlink, celda.value, celda.style = celda.value, "Abrir mapa", "Hyperlink"

def resumen(df):
    if df.empty:
        return pd.DataFrame()
    t = pd.pivot_table(df, index=["Categoría", "Subtema"], columns="Banda de tiempo", values="Nombre",
                       aggfunc="count", fill_value=0)
    t["Total"] = t.sum(axis=1)
    t["Tiempo mínimo (min)"] = df.groupby(["Categoría", "Subtema"])["Tiempo (min)"].min().round(1)
    return t.reset_index()

def exportar_excel(df, ruta, parametros):
    with pd.ExcelWriter(ruta, engine="openpyxl") as xw:
        pd.DataFrame(parametros.items(), columns=["Parámetro", "Valor"]).to_excel(xw, sheet_name="0_Metodologia", index=False)
        _formatear(xw.sheets["0_Metodologia"], pd.DataFrame(columns=["Parámetro", "Valor"]))
        xw.sheets["0_Metodologia"].column_dimensions["B"].width = 110
        r = resumen(df)
        r.to_excel(xw, sheet_name="Resumen", index=False)
        _formatear(xw.sheets["Resumen"], r)
        for cat in CATEGORIAS:
            sub = df[df["Categoría"] == cat] if not df.empty else df
            cols = COLUMNAS_BASE[:4] + COLUMNAS_CAT[cat] + COLUMNAS_BASE[4:]
            sub = sub.reindex(columns=cols + ["Color subtema"])
            colores = list(sub.pop("Color subtema").fillna("#999999"))
            sub.to_excel(xw, sheet_name=HOJAS[cat], index=False)
            _formatear(xw.sheets[HOJAS[cat]], sub, colores)
    print("Excel generado:", ruta)

def exportar_comparativo(df, ruta):
    if df.empty:
        return
    t = pd.pivot_table(df, index=["Categoría", "Subtema"], columns="Ubicación estudio", values="Nombre",
                       aggfunc="count", fill_value=0).reset_index()
    m = pd.pivot_table(df, index="Categoría", columns="Ubicación estudio", values="Tiempo (min)",
                       aggfunc=["count", "min", "median"]).round(1)
    m.columns = [f"{a} – {b}".replace("count", "Total").replace("min", "Tiempo mín").replace("median", "Tiempo mediana")
                 for a, b in m.columns]
    m = m.reset_index()
    with pd.ExcelWriter(ruta, engine="openpyxl") as xw:
        t.to_excel(xw, sheet_name="Comparativo_subtemas", index=False); _formatear(xw.sheets["Comparativo_subtemas"], t)
        m.to_excel(xw, sheet_name="Comparativo_categorias", index=False); _formatear(xw.sheets["Comparativo_categorias"], m)
        for ubi, g in df.groupby("Ubicación estudio"):
            g = g.reindex(columns=COLUMNAS_BASE)
            g.to_excel(xw, sheet_name=f"Detalle_{ubi}"[:31], index=False); _formatear(xw.sheets[f"Detalle_{ubi}"[:31]], g)
    print("Excel comparativo generado:", ruta)

In [ ]:
PARAMETROS = {
    "Fuente de establecimientos": "DENUE (INEGI) + OpenStreetMap (Overpass API)",
    "Tiempo de traslado": f"OSRM (auto, sin tráfico) × factor {FACTOR_TRAFICO}",
    "Sentido del traslado": "Desde cada establecimiento hacia el sitio propuesto del CAM",
    "Zona de influencia": f"≤ {TIEMPO_MAX_MIN} minutos (bandas {BANDAS_MIN})",
    "Fecha de extracción": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
    "Número de médicos": "DENUE publica el estrato de personal ocupado (proxy); validar en campo",
    "Horarios": "Solo cuando OSM los publica; DENUE no los incluye",
    "Subtemas": "Clasificación automática por nombre y clase SCIAN; revisar casos dudosos",
}
for nombre_ubi in UBICACIONES:
    u = UBICACIONES[nombre_ubi]
    p = dict(PARAMETROS, **{"Ubicación": f"{nombre_ubi} ({u['lat']:.6f}, {u['lng']:.6f})", "Enlace": u["url"]})
    exportar_excel(datos[datos["Ubicación estudio"] == nombre_ubi], SALIDA / f"CAM_{norm(nombre_ubi).strip().replace(' ', '_')}_gratuito.xlsx", p)
exportar_comparativo(datos, SALIDA / "CAM_comparativo_gratuito.xlsx")

## 6. Zona de influencia por tiempo (isócrona aproximada)

In [ ]:
# Zona de influencia real (isócrona aproximada): cada celda de la malla se colorea por su tiempo al sitio.
isocronas = {}
if ISOCRONA:
    for nombre_ubi, u in UBICACIONES.items():
        pts = malla(u["lat"], u["lng"], RADIO_BUSQUEDA_M, PASO_MALLA_M)
        t = tiempos_osrm(pts, (u["lat"], u["lng"]))
        isocronas[nombre_ubi] = [(la, ln, m) for (la, ln), (m, _) in zip(pts, t) if m is not None and m <= TIEMPO_MAX_MIN]
        print(f"{nombre_ubi}: {len(isocronas[nombre_ubi])} celdas dentro de {TIEMPO_MAX_MIN} min")

## 7. Mapas (5 por ubicación)
★ = sitio propuesto · **icono** = categoría (🏥 casa médica, 🩺 médico, 🧪 matraz, 🩻 rayos X) · **color** = subtema · fondo verde/amarillo/naranja = banda de tiempo. Usa el control de capas (esquina superior derecha) para filtrar subtemas.

In [ ]:
import folium
from folium.plugins import BeautifyIcon, Fullscreen, MiniMap
from branca.element import Template, MacroElement

COLOR_BANDA = ["#1a9850", "#fee08b", "#f46d43", "#a50026"]

def leyenda(df):
    filas = ""
    for cat in CATEGORIAS:
        subs = df[df["Categoría"] == cat]["Subtema"].value_counts()
        if subs.empty:
            continue
        filas += f"<div style='margin-top:6px'><i class='fa fa-{ICONOS[cat]}'></i> <b>{cat}</b></div>"
        for s, n in subs.items():
            filas += (f"<div><span style='display:inline-block;width:12px;height:12px;border-radius:50%;"
                      f"background:{SUBTEMAS[cat].get(s, '#999')}'></span> {s} ({n})</div>")
    if ISOCRONA:
        ini = 0
        filas += "<div style='margin-top:6px'><b>Tiempo al sitio</b></div>"
        for fin, c in zip(BANDAS_MIN, COLOR_BANDA):
            filas += f"<div><span style='display:inline-block;width:12px;height:12px;background:{c};opacity:.6'></span> {ini}-{fin} min</div>"
            ini = fin
    tpl = Template("{% macro html(this, kwargs) %}<div style='position:fixed;bottom:20px;left:20px;z-index:9999;"
                   "background:white;padding:8px 10px;border-radius:6px;box-shadow:0 1px 5px rgba(0,0,0,.4);"
                   "font-size:12px;max-height:60%;overflow:auto'><b>Leyenda</b>" + filas + "</div>{% endmacro %}")
    m = MacroElement(); m._template = tpl
    return m

def popup(r):
    campos = ["Categoría", "Subtema", "Dirección", "Tiempo (min)", "Distancia ruta (km)", "Horarios", "Teléfono", "Fuente"]
    campos += COLUMNAS_CAT[r["Categoría"]][:-1]
    filas = "".join(f"<tr><td><b>{c}</b></td><td>{html.escape(str(r.get(c, '')))}</td></tr>" for c in campos)
    return folium.Popup(f"<h5>{html.escape(r['Nombre'])}</h5><table>{filas}</table>"
                        f"<a href='{r['Link mapa']}' target='_blank'>Ver en mapa</a>", max_width=360)

def mapa_folium(nombre_ubi, df, titulo, archivo):
    u = UBICACIONES[nombre_ubi]
    m = folium.Map(location=[u["lat"], u["lng"]], zoom_start=12, tiles="OpenStreetMap", control_scale=True)
    folium.TileLayer("CartoDB positron", name="Mapa claro").add_to(m)
    if ISOCRONA and isocronas.get(nombre_ubi):
        capa = folium.FeatureGroup(name=f"Zona ≤ {TIEMPO_MAX_MIN} min", show=True)
        dlat = PASO_MALLA_M / 111_320 / 2
        for la, ln, mins in isocronas[nombre_ubi]:
            dlng = PASO_MALLA_M / (111_320 * math.cos(math.radians(la))) / 2
            c = COLOR_BANDA[next(i for i, b in enumerate(BANDAS_MIN) if mins <= b)]
            folium.Rectangle([[la - dlat, ln - dlng], [la + dlat, ln + dlng]], fill=True, stroke=False,
                             fill_color=c, fill_opacity=.18, tooltip=f"{mins:.0f} min").add_to(capa)
        capa.add_to(m)
    folium.Marker([u["lat"], u["lng"]], tooltip=f"Sitio propuesto CAM – {nombre_ubi}",
                  icon=folium.Icon(color="red", icon="star", prefix="fa")).add_to(m)
    for (cat, sub), g in df.groupby(["Categoría", "Subtema"]):
        capa = folium.FeatureGroup(name=f"{cat} · {sub} ({len(g)})")
        for _, r in g.iterrows():
            icono = BeautifyIcon(icon=ICONOS[cat], icon_shape="marker", background_color=r["Color subtema"],
                                 border_color="#333", text_color="white", prefix="fa")
            folium.Marker([r["Latitud"], r["Longitud"]], icon=icono, popup=popup(r),
                          tooltip=f"{r['Nombre']} · {r['Tiempo (min)']:.0f} min").add_to(capa)
        capa.add_to(m)
    m.get_root().html.add_child(folium.Element(
        f"<h4 style='position:fixed;top:8px;left:55px;z-index:9999;background:white;padding:4px 8px;"
        f"border-radius:4px;box-shadow:0 1px 4px rgba(0,0,0,.3)'>{html.escape(titulo)}</h4>"))
    m.get_root().add_child(leyenda(df))
    folium.LayerControl(collapsed=True).add_to(m)
    Fullscreen().add_to(m); MiniMap(toggle_display=True).add_to(m)
    m.save(SALIDA / archivo)
    return m

# 5 mapas por ubicación: 1 general + 1 por cada categoría del estudio
mapas = {}
for nombre_ubi in UBICACIONES:
    d = datos[datos["Ubicación estudio"] == nombre_ubi]
    slug = norm(nombre_ubi).strip().replace(" ", "_")
    mapas[(nombre_ubi, "General")] = mapa_folium(nombre_ubi, d, f"{nombre_ubi} – Todos", f"{slug}_0_general.html")
    for i, cat in enumerate(CATEGORIAS, 1):
        mapas[(nombre_ubi, cat)] = mapa_folium(nombre_ubi, d[d["Categoría"] == cat], f"{nombre_ubi} – {cat}", f"{slug}_{i}_{norm(cat).split()[0]}.html")
print("Mapas guardados en", SALIDA.resolve())
mapas[(list(UBICACIONES)[0], "General")]

In [ ]:
# Mostrar cualquier otro mapa, p. ej. laboratorios de San José:
mapas[("San José", CAT_LAB)]

## 8. Vista rápida de resultados

In [ ]:
datos[COLUMNAS_BASE[:8]].head(30)